In [3]:
import requests
import pandas as pd


# Get top 100 games by player count from SteamSpy
response = requests.get('https://steamspy.com/api.php?request=top100in2weeks')
top_games = pd.DataFrame(response.json()).T #shortand for JSON transpose rows to cols
top_games = top_games.head(100) #get first 100 rows
top_games['appid'] = top_games['appid'].astype(int) #convert appid to integer
top_games[['appid', 'name']].head() #get first 5 rows of appid and name

,appid,name
730,730,Counter-Strike: Global Offensive
1172470,1172470,Apex Legends
578080,578080,PUBG: BATTLEGROUNDS
1623730,1623730,Palworld
440,440,Team Fortress 2


In [4]:
from dotenv import load_dotenv
from steam.webapi import WebAPI
import os

load_dotenv()

STEAM_API_KEY = os.getenv('STEAM_API_KEY')
api = WebAPI(key=STEAM_API_KEY)

# Example: Get reviews for a single app
def get_reviews(appid, num_reviews=100):
    url = f"https://store.steampowered.com/appreviews/{appid}"
    params = {
        'json': 1,
        'num_per_page': num_reviews,
        'filter': 'recent',
        'language': 'all'
    }
    r = requests.get(url, params=params)
    if r.status_code == 200:
        data = r.json()
        if 'reviews' in data:
            return data['reviews']
    return []

In [5]:
from tqdm import tqdm

all_reviews = []
for _, row in tqdm(top_games.iterrows(), total=top_games.shape[0]):
    appid = row['appid']
    name = row['name']
    reviews = get_reviews(appid, num_reviews=100)
    for review in reviews:
        all_reviews.append({
            'appid': appid,
            'name': name,
            'review': review['review'],
            'timestamp_created': review['timestamp_created'],
            'voted_up': review['voted_up'],
            'votes_up': review['votes_up'],
            'votes_funny': review['votes_funny'],
            'weighted_vote_score': review['weighted_vote_score'],
        })

reviews_df = pd.DataFrame(all_reviews)
reviews_df.head()

100%|██████████| 100/100 [00:56<00:00,  1.78it/s]


,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
0,730,Counter-Strike: Global Offensive,I love this toxic goofy ass game,1751624095,True,0,0,0.5
1,730,Counter-Strike: Global Offensive,Gut diese,1751624061,True,0,0,0.5
2,730,Counter-Strike: Global Offensive,"too many hacker in this game, after report the...",1751624028,False,0,0,0.5
3,730,Counter-Strike: Global Offensive,go go go出发咯,1751624014,True,0,0,0.5
4,730,Counter-Strike: Global Offensive,好玩,1751623947,True,0,0,0.5
